In [1]:
%pip install sentence-transformers scikit-learn matplotlib pandas numpy torch

   ---------------------------------------- 0.0/570.8 kB ? eta -:--:--
   ---------------------------------------- 570.8/570.8 kB 6.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import re
import json
import random
import hashlib
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

import torch
from sentence_transformers import SentenceTransformer




SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "all-MiniLM-L6-v2"

def detect_base_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "homeworks" / "HW14",
        Path.cwd().parent,
        Path.cwd().parent / "homeworks" / "HW14",
    ]
    for c in candidates:
        if (c / "data" / "kb.json").exists():
            return c
    raise FileNotFoundError("Не найден файл data/kb.json. Положи датасет в homeworks/HW14/data/kb.json")

BASE_DIR = detect_base_dir()
DATA_PATH = BASE_DIR / "data" / "kb.json"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR.resolve())
print("DATA_PATH:", DATA_PATH.resolve())
print("ARTIFACTS_DIR:", ARTIFACTS_DIR.resolve())
print("Устройство:", DEVICE)
print("Модель эмбеддингов:", MODEL_NAME)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

def extract_items(raw):
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        for key in ["data", "items", "documents", "docs", "records", "entries", "articles"]:
            if key in raw and isinstance(raw[key], list):
                return raw[key]
        return [raw]
    return [raw]

items = extract_items(raw_data)

def pick_text(d: dict):
    text_keys = ["text", "content", "body", "article", "document", "paragraph", "description", "answer"]
    title_keys = ["title", "headline", "subject", "question", "name"]

    title = None
    text = None

    for k in title_keys:
        if k in d and isinstance(d[k], str) and d[k].strip():
            title = d[k].strip()
            break

    for k in text_keys:
        if k in d and isinstance(d[k], str) and d[k].strip():
            text = d[k].strip()
            break

    if text is None:
        vals = []
        for v in d.values():
            if isinstance(v, str) and v.strip():
                vals.append(v.strip())
            elif isinstance(v, list):
                for x in v:
                    if isinstance(x, str) and x.strip():
                        vals.append(x.strip())
        if vals:
            text = max(vals, key=len)
            if title is None:
                title = vals[0][:80]

    if text is None:
        return None, None

    if title is None:
        title = text[:80]

    return text, title

def normalize_documents(items):
    docs = []
    for i, item in enumerate(items, start=1):
        if isinstance(item, str):
            text = re.sub(r"\s+", " ", item).strip()
            if len(text) < 20:
                continue
            docs.append({
                "doc_id": f"doc_{i:03d}",
                "title": text[:80],
                "text": text
            })

        elif isinstance(item, dict):
            text, title = pick_text(item)
            if text is None:
                continue

            text = re.sub(r"\s+", " ", text).strip()
            title = re.sub(r"\s+", " ", str(title)).strip()

            if len(text) < 20:
                continue

            doc_id = item.get("doc_id") or item.get("id") or item.get("uid") or f"doc_{i:03d}"

            docs.append({
                "doc_id": str(doc_id),
                "title": title[:120],
                "text": text
            })

    return docs

documents = normalize_documents(items)

if len(documents) < 8:
    raise ValueError("После нормализации получилось слишком мало документов. Нужен kb.json хотя бы с 8-10 текстами.")

print("\nЧисло документов после нормализации:", len(documents))
print("\nПримеры документов:")
for i, doc in enumerate(documents[:3], start=1):
    print(f"\n{i}. {doc['doc_id']} — {doc['title']}")
    print(textwrap.shorten(doc["text"], width=220, placeholder=" ..."))



def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text.strip())
    return text

def chunk_text(text: str, chunk_size: int = 80, overlap: int = 20):
    words = clean_text(text).split()
    if len(words) <= chunk_size:
        return [" ".join(words)]
    chunks = []
    start = 0
    step = max(1, chunk_size - overlap)
    while start < len(words):
        end = start + chunk_size
        piece = words[start:end]
        if not piece:
            break
        chunks.append(" ".join(piece))
        if end >= len(words):
            break
        start += step
    return chunks

def documents_to_chunks(docs, chunk_size=80, overlap=20):
    rows = []
    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for i, chunk in enumerate(chunks, start=1):
            rows.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_id": f"{doc['doc_id']}_chunk_{i}",
                "chunk_index": i,
                "chunk_text": chunk
            })
    return pd.DataFrame(rows)



print("\nЗагрузка модели...")
embedder = SentenceTransformer(MODEL_NAME, device=DEVICE)
print("Модель загружена.")

def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

def texts_fingerprint(texts, suffix=""):
    h = hashlib.sha256((MODEL_NAME + suffix).encode("utf-8"))
    for t in texts:
        h.update(t.encode("utf-8", errors="replace"))
        h.update(b"\xff")
    return h.hexdigest()[:24]

def encode_texts(texts, cache_label="base"):
    cache_name = f"emb_{cache_label}_{texts_fingerprint(texts, cache_label)}.npz"
    cache_path = ARTIFACTS_DIR / cache_name
    if cache_path.exists():
        return np.load(cache_path)["emb"].astype("float32")

    emb = embedder.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        show_progress_bar=len(texts) > 8,
        normalize_embeddings=False,
    ).astype("float32")

    emb = l2_normalize(emb)
    np.savez_compressed(cache_path, emb=emb)
    return emb

def dense_search(query: str, emb_matrix: np.ndarray, chunks_df: pd.DataFrame, top_k: int = 3):
    q = embedder.encode(
        [query],
        batch_size=1,
        convert_to_numpy=True,
        show_progress_bar=False,
        normalize_embeddings=False,
    ).astype("float32")
    q = l2_normalize(q)
    sims = cosine_similarity(q, emb_matrix)[0]
    idx = np.argsort(-sims)[:top_k]
    scores = sims[idx]
    out = chunks_df.iloc[idx].copy().reset_index(drop=True)
    out["score"] = scores
    return out



def first_sentence(text):
    parts = re.split(r"(?<=[.!?])\s+", clean_text(text))
    for p in parts:
        if len(p.split()) >= 4:
            return p
    return clean_text(text)[:120]

def build_control_queries(docs, n=10):
    selected = docs[:n]
    queries = []
    for doc in selected:
        title = clean_text(doc["title"])
        query = title if len(title.split()) >= 3 else first_sentence(doc["text"])
        queries.append({
            "query": query,
            "expected_source": doc["doc_id"]
        })
    return queries

control_queries = build_control_queries(documents, n=min(10, len(documents)))



def evaluate_retrieval(queries, emb_matrix, chunks_df, top_k=3):
    rows = []
    for item in queries:
        res = dense_search(item["query"], emb_matrix, chunks_df, top_k=top_k)
        retrieved_sources = res["doc_id"].tolist()
        expected = item["expected_source"]
        hit = int(expected in retrieved_sources)
        recall = 1.0 if hit else 0.0
        rank = ""
        mrr = 0.0

        for i, src in enumerate(retrieved_sources, start=1):
            if src == expected:
                rank = i
                mrr = 1.0 / i
                break

        rows.append({
            "query": item["query"],
            "expected_source": expected,
            "retrieved_sources": " | ".join(retrieved_sources),
            "hit_at_k": hit,
            "recall_at_k": recall,
            "rank_of_first_relevant": rank,
            "mrr_at_k": mrr,
        })

    df = pd.DataFrame(rows)
    metrics = {
        "hit_at_k": float(df["hit_at_k"].mean()),
        "recall_at_k": float(df["recall_at_k"].mean()),
        "mrr_at_k": float(df["mrr_at_k"].mean()),
        "n_queries": int(len(df)),
        "top_k": int(top_k),
    }
    return df, metrics



def compare_chunk_sizes(docs, queries, chunk_sizes=(60, 90), overlap=20, top_k=3):
    rows = []
    summary = []

    for cs in chunk_sizes:
        cdf = documents_to_chunks(docs, chunk_size=cs, overlap=overlap)
        emb = encode_texts(cdf["chunk_text"].tolist(), cache_label=f"chunk_{cs}")
        eval_df, metrics = evaluate_retrieval(queries, emb, cdf, top_k=top_k)

        summary.append({
            "chunk_size": cs,
            "n_chunks": len(cdf),
            "hit_at_k": metrics["hit_at_k"],
            "recall_at_k": metrics["recall_at_k"],
            "mrr_at_k": metrics["mrr_at_k"],
        })

        for _, r in eval_df.iterrows():
            rows.append({
                "chunk_size": cs,
                "query": r["query"],
                "expected_source": r["expected_source"],
                "retrieved_sources": r["retrieved_sources"],
                "hit_at_k": r["hit_at_k"],
                "recall_at_k": r["recall_at_k"],
                "rank_of_first_relevant": r["rank_of_first_relevant"],
                "mrr_at_k": r["mrr_at_k"],
            })

    comp_df = pd.DataFrame(rows)
    summary_df = pd.DataFrame(summary).sort_values(
        by=["hit_at_k", "mrr_at_k", "recall_at_k", "chunk_size"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

    best_chunk_size = int(summary_df.iloc[0]["chunk_size"])
    return comp_df, summary_df, best_chunk_size

print("\n=== Эксперимент по chunk_size ===")
comparison_df, comparison_summary_df, best_chunk_size = compare_chunk_sizes(
    documents,
    control_queries,
    chunk_sizes=(60, 90),
    overlap=20,
    top_k=3
)
print(comparison_summary_df.to_string(index=False))

CHUNK_SIZE = best_chunk_size
OVERLAP = 20
TOP_K = 3

chunks_df = documents_to_chunks(documents, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
chunk_emb = encode_texts(chunks_df["chunk_text"].tolist(), cache_label="base_final")

print("\nЧисло чанков:", len(chunks_df))
print("\nПример чанков:")
print(chunks_df.head(5).to_string(index=False))


print("\n=== Примеры retrieval ===")
for q in [x["query"] for x in control_queries[:3]]:
    print("\nЗапрос:", q)
    res = dense_search(q, chunk_emb, chunks_df, top_k=TOP_K)
    print(res[["doc_id", "title", "score"]].to_string(index=False))



print("\n=== Оценка retrieval ===")
retrieval_eval_df, retrieval_metrics = evaluate_retrieval(control_queries, chunk_emb, chunks_df, top_k=TOP_K)
print(retrieval_eval_df.to_string(index=False))
print("\nМетрики:")
for k, v in retrieval_metrics.items():
    print(f"{k}: {v}")

retrieval_eval_df.to_csv(ARTIFACTS_DIR / "retrieval_eval.csv", index=False, encoding="utf-8")



new_docs = [
    {
        "doc_id": "update_001",
        "title": "Правила бронирования GPU",
        "text": "Для работы с учебным GPU-сервером требуется предварительное бронирование слота. Один слот длится 30 минут. При перегрузке видеопамяти необходимо уменьшить batch size. При систематическом нарушении расписания доступ к GPU может быть временно ограничен."
    },
    {
        "doc_id": "update_002",
        "title": "Политика fair-share",
        "text": "Политика fair-share снижает приоритет слишком длинных задач, чтобы короткие учебные эксперименты не простаивали в очереди. Фоновые задачи без учебного тега переносятся на ночные окна. Это позволяет распределять вычислительные ресурсы между студентами более равномерно."
    },
    {
        "doc_id": "update_003",
        "title": "Что такое OOM на GPU",
        "text": "OOM означает нехватку видеопамяти. При OOM рекомендуется уменьшить batch size, упростить модель или сократить длину последовательности. После ошибки процесс нужно завершить, чтобы освободить память для других пользователей."
    },
]

update_queries = [
    "Как бронировать слот на GPU?",
    "Что делает политика fair-share?",
    "Что означает OOM на GPU?"
]

before_after_rows = []
for q in update_queries:
    before = dense_search(q, chunk_emb, chunks_df, top_k=TOP_K)
    before_sources = before["doc_id"].tolist()
    before_after_rows.append({
        "query": q,
        "before_retrieved_sources": " | ".join(before_sources),
        "after_retrieved_sources": "",
        "changed": "",
    })

updated_documents = documents + new_docs
chunks_df_updated = documents_to_chunks(updated_documents, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
chunk_emb_updated = encode_texts(chunks_df_updated["chunk_text"].tolist(), cache_label="updated_final")

for i, q in enumerate(update_queries):
    after = dense_search(q, chunk_emb_updated, chunks_df_updated, top_k=TOP_K)
    after_sources = after["doc_id"].tolist()
    before_after_rows[i]["after_retrieved_sources"] = " | ".join(after_sources)
    before_after_rows[i]["changed"] = str(before_after_rows[i]["before_retrieved_sources"] != before_after_rows[i]["after_retrieved_sources"])

before_after_df = pd.DataFrame(before_after_rows)
print("\n=== До/после обновления ===")
print(before_after_df.to_string(index=False))
before_after_df.to_csv(ARTIFACTS_DIR / "retrieval_before_after_update.csv", index=False, encoding="utf-8")


def simple_answer_from_context(question, retrieved_df, max_sentences=2):
    query_terms = set(re.findall(r"[a-zA-Zа-яА-ЯёЁ0-9]+", question.lower()))
    candidates = []

    for _, row in retrieved_df.iterrows():
        for sent in re.split(r"(?<=[.!?])\s+", row["chunk_text"]):
            sent = sent.strip()
            if not sent:
                continue
            sent_terms = set(re.findall(r"[a-zA-Zа-яА-ЯёЁ0-9]+", sent.lower()))
            overlap_score = len(query_terms & sent_terms)
            score = overlap_score + float(row["score"])
            candidates.append((score, sent, row["doc_id"]))

    candidates.sort(key=lambda x: x[0], reverse=True)

    selected = []
    seen = set()
    for score, sent, src in candidates:
        key = sent.lower()
        if key not in seen:
            selected.append((sent, src))
            seen.add(key)
        if len(selected) >= max_sentences:
            break

    if not selected:
        return "Недостаточно подходящего контекста для уверенного ответа.", []

    answer = " ".join([x[0] for x in selected])
    sources = []
    for _, src in selected:
        if src not in sources:
            sources.append(src)

    return answer, sources

def run_mini_rag(questions, emb_matrix, chunks_df, top_k=3):
    rows = []
    for q in questions:
        retrieved = dense_search(q, emb_matrix, chunks_df, top_k=top_k)
        answer, answer_sources = simple_answer_from_context(q, retrieved, max_sentences=2)
        rows.append({
            "question": q,
            "answer": answer,
            "retrieved_sources": " | ".join(retrieved["doc_id"].tolist()),
            "answer_sources": " | ".join(answer_sources),
        })
    return pd.DataFrame(rows)

rag_questions = [
    control_queries[0]["query"],
    control_queries[1]["query"],
    "Как бронировать слот на GPU?",
    "Что означает OOM на GPU?",
    "Что делает политика fair-share?"
]

rag_examples_df = run_mini_rag(rag_questions, chunk_emb_updated, chunks_df_updated, top_k=TOP_K)
print("\n=== Mini-RAG ===")
print(rag_examples_df.to_string(index=False))
rag_examples_df.to_csv(ARTIFACTS_DIR / "rag_examples.csv", index=False, encoding="utf-8")


plt.figure(figsize=(8, 4))
plt.bar(comparison_summary_df["chunk_size"].astype(str), comparison_summary_df["hit_at_k"])
plt.title("Сравнение hit@k для разных chunk_size")
plt.xlabel("chunk_size")
plt.ylabel("hit@k")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "retrieval_quality_plot.png", dpi=150, bbox_inches="tight")
plt.close()

summary_json = {
    "seed": SEED,
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "n_documents_before_update": len(documents),
    "n_documents_after_update": len(updated_documents),
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "top_k": TOP_K,
    "n_chunks_before_update": int(len(chunks_df)),
    "n_chunks_after_update": int(len(chunks_df_updated)),
    "retrieval_metrics": retrieval_metrics,
    "experiment_summary": comparison_summary_df.to_dict(orient="records"),
}

with open(ARTIFACTS_DIR / "retrieval_metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)



print("\n=== Сохранённые файлы ===")
for p in [
    ARTIFACTS_DIR / "retrieval_eval.csv",
    ARTIFACTS_DIR / "rag_examples.csv",
    ARTIFACTS_DIR / "retrieval_before_after_update.csv",
    ARTIFACTS_DIR / "retrieval_metrics_summary.json",
    ARTIFACTS_DIR / "retrieval_quality_plot.png",
]:
    print(p.as_posix())

print("\nГотово.")

BASE_DIR: C:\Users\kiril\OneDrive\Desktop\ИИИ\REP\III_2025-26\homeworks\HW14
DATA_PATH: C:\Users\kiril\OneDrive\Desktop\ИИИ\REP\III_2025-26\homeworks\HW14\data\kb.json
ARTIFACTS_DIR: C:\Users\kiril\OneDrive\Desktop\ИИИ\REP\III_2025-26\homeworks\HW14\artifacts
Устройство: cpu
Модель эмбеддингов: all-MiniLM-L6-v2

Число документов после нормализации: 12

Примеры документов:

1. sec_01 — Надёжные пароли и менеджеры паролей
Надёжный пароль должен быть длинным и уникальным для каждого сервиса. Повторное использование одного и того же пароля на разных сайтах опасно, потому что утечка на одном ресурсе открывает доступ и к другим аккаунтам. ...

2. sec_02 — Двухфакторная аутентификация
Двухфакторная аутентификация добавляет второй фактор помимо пароля. Это может быть приложение-аутентификатор, аппаратный ключ или одноразовый код. Если пароль украден, второй фактор заметно снижает риск захвата ...

3. sec_03 — Фишинг и признаки поддельных сообщений
Фишинг — это попытка заставить пользователя ра

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10828.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена.

=== Эксперимент по chunk_size ===
 chunk_size  n_chunks  hit_at_k  recall_at_k  mrr_at_k
         90        12       1.0          1.0  0.850000
         60        13       1.0          1.0  0.566667

Число чанков: 12

Пример чанков:
doc_id                                  title       chunk_id  chunk_index                                                                                                                                                                                                                                                                                                                                                                                                                                chunk_text
sec_01    Надёжные пароли и менеджеры паролей sec_01_chunk_1            1 Надёжный пароль должен быть длинным и уникальным для каждого сервиса. Повторное использование одного и того же пароля на разных сайтах опасно, потому что утечка на одном ресу